<div style="text-align: justify; max-width: 90ch">

# Lecture 03 exercises (solutions)

Exercises are tagged with the goal they practise (see `reading_material/goals_03.md`); `[O1]` marks optional exercises on the career-track notebook `lec_03e`.
Time bands:

- 15 min (quick)
- 45 min (realistic)
- 90 min

Before you start:

- Run the configuration cell first, then the **Rebuild inputs** cell.
- Ollama must be running with the models pulled.
- The MLflow server must be started (guide `03a_mlflow_server`).
- `lec_02d` must have run once: the exercises reuse its Chroma index (otherwise the Rebuild inputs cell builds it, which takes minutes).

</div>

In [1]:
import os
import time
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from mlflow.entities import SpanType
from mlflow.genai.scorers import scorer

from course_helpers.agents import (
    GRADE_PROMPT,
    GRADE_SCHEMA,
    make_agentic_rag,
    make_course_tools,
    message_table,
    run_tool_calls,
    searches,
    total_tokens,
)
from course_helpers.checks import check_mlflow
from course_helpers.corpus import load_corpus, load_eval_set
from course_helpers.evaluation import (
    DETERMINISTIC_SCORERS,
    compare_variants,
    fact_keywords_found,
    is_refusal,
    judge_input_tokens,
    make_facts_judge,
    measure_agreement_between,
    per_question_table,
    prompt_tokens,
    run_judge_on,
    build_eval_records_from,
)
from course_helpers.rag import (
    LECTURE_KINDS,
    build_chroma_index,
    format_context_from,
    make_rag_chain,
    split_documents,
)

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-03-exercises"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from course_helpers.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs** (given, run it once): the system of `lec_03a`-`lec_03b` from the `course_helpers` helpers, the notes as `Document`s for the tools of `lec_03d`,
the evaluation set by question id, and tracing into the experiment `llm-course-03-exercises`. The chat model is that of `lec_03b`:
`num_predict=200` and no `num_ctx`, so it shares Ollama's default window with the judge and the model is not reloaded.

</div>

In [3]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

documents = load_corpus(REPO / "corpus" / "uoa_py_course")
store = build_chroma_index(
    split_documents(documents, chunk_size=800, chunk_overlap=100),
    "course_notes",
    DATA_DIR / "chroma",
    EMBED_MODEL,
    OLLAMA_HOST,
)
retriever = store.as_retriever(
    search_kwargs={"k": 4, "filter": {"kind": {"$in": LECTURE_KINDS}}}
)

llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=200,
)
rag_chain = make_rag_chain(retriever, llm)

eval_set = load_eval_set(REPO / "corpus" / "eval" / "qa_eval_set.jsonl")
eval_by_id = {row["id"]: row for row in eval_set}

check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

print(f"{len(eval_set)} questions in the evaluation set")

Ollama OK at http://localhost:11434; models ready: nomic-embed-text


2026/10/12 19:08:50 INFO mlflow.tracking.fluent: Experiment with name 'llm-course-03-exercises' does not exist. Creating a new experiment.


Reusing 'course_notes': 2452 chunks in chroma/
MLflow OK at http://127.0.0.1:5010
19 questions in the evaluation set


<div style="text-align: justify; max-width: 90ch">

### Exercise 3.1 [G1, G3] All 19 questions, aggregates against rows (45 min)

Turn the whole evaluation set into MLflow records (`build_eval_records_from`) and evaluate the RAG chain of the Rebuild inputs cell on all 19 questions with the deterministic scorers of `lec_03a` (`DETERMINISTIC_SCORERS`),
in one named run. Print the aggregate metrics, then the per-question table (`per_question_table`).

- Which scorer's mean hides the most failing rows?
- For two failing rows, read the answer (in the MLflow UI or in `results.result_df["response"]`) and decide: is the answer wrong, or only the check?

</div>

In [4]:
records_19 = build_eval_records_from(eval_set)


def predict(question):
    """The system under test: the RAG chain of the Rebuild inputs cell (lec_03b's settings)."""
    return rag_chain.invoke(question)


with mlflow.start_run(run_name="3.1 all 19, deterministic"):
    results_19 = mlflow.genai.evaluate(
        data=records_19, predict_fn=predict, scorers=DETERMINISTIC_SCORERS
    )

pd.Series(results_19.metrics).sort_index().round(2)

2026/10/12 19:08:52 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


2026/10/12 19:08:52 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


answer_tokens/max             200.00
answer_tokens/mean            167.89
cites_expected_source/mean      0.37
fact_keywords_found/mean        0.85
refused/mean                    0.00
retrieval_hit/mean              1.00
dtype: float64

In [5]:
table_19 = per_question_table(results_19, eval_set, DETERMINISTIC_SCORERS)

table_19

,cites_expected_source,retrieval_hit,answer_tokens,refused,fact_keywords_found
id,,,,,
q01,True,True,106.0,False,1.000000
q02,True,True,200.0,False,1.000000
q03,True,True,115.0,False,1.000000
q04,False,True,200.0,False,0.500000
q05,False,True,200.0,False,1.000000
q06,False,True,200.0,False,0.500000
q07,True,True,200.0,False,1.000000
q08,True,True,173.0,False,1.000000
q09,False,True,200.0,False,1.000000


In [6]:
def question_of(request):
    """The question of a result row: text, or {"question": ...}, depending on the trace."""
    return request["question"] if isinstance(request, dict) else request


id_of = {row["question"]: row["id"] for row in eval_set}
answers_19 = {
    id_of[question_of(request)]: response
    for request, response in zip(
        results_19.result_df["request"], results_19.result_df["response"]
    )
}

# The end of every answer that does not name its file: is the source line missing or cut off?
for qid in table_19.index[table_19["cites_expected_source"].eq(False)]:
    tokens = table_19.loc[qid, "answer_tokens"]
    print(f"{qid} ({tokens} tokens): ...{answers_19[qid][-110:]!r}")

q04 (200.0 tokens): ...'un context.\n- You can log multiple metrics in a single call.\n\n### Source File:\n- **`[source: lecture_13/lec_13'
q05 (200.0 tokens): ...'- The **nested runs** allow for logging of individual configurations and metrics.\n\n### Source File:\n`lecture_1'
q06 (200.0 tokens): ...' serve -m "models:/heart_disease_clf@champion" --port 5002 --no-conda\n   ```\n\n### Explanation:\n- `-m "models:/'
q09 (200.0 tokens): ...'ro, leading to feature selection.\n\n**Answer:** Ridge regularization shrinks coefficients towards zero but does'
q10 (200.0 tokens): ...' much larger than the distance feature (in meters), and the model will give more weight to the height feature,'
q11 (123.0 tokens): ...' sensitive model to local noise or outliers.\n\n**Source file:** `lecture_10/lec_10c_knn_decision_boundaries.md`'
q12 (200.0 tokens): ...'oid function is:\n   $$\n   p = \\frac{1}{1 + \\exp(-\\text{logit}(p))} = \\frac{1}{1 + \\exp(-(\\beta_0 + \\beta_1 x_1'
q14 (200.0 tokens): ...'

<div style="text-align: justify; max-width: 90ch">

**Example answer.** `retrieval_hit/mean` is at or near 1.0: the right file reached the prompt for (almost) every question,
and that row is the same on every run, because the search is deterministic. The mean that hides the most is `cites_expected_source`:
in example runs it was between about 0.4 and 0.6, eight to twelve failing rows behind one number. The last cell shows why:
compare each `answer_tokens` with 200; in example runs most failing rows had used all 200. An answer that used all 200 tokens was cut off by `num_predict` before its source line,
so the answer is not wrong, the system stopped it (exercise 3.5 tests 400). A failing row with fewer tokens names no file or another file (in one example run,
q11 named `lec_10a` instead of `lec_10d`): there the check is right. Rows with `fact_keywords_found` below 1 need a reader;
exercise 3.4 hands them to the judge.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.2 [G2] Find the failing step from the trace (15 min)

Ask the RAG chain of the Rebuild inputs cell q09 of the evaluation set, "What is the difference between Ridge and Lasso regularization?".
From its trace, print the sources in the RETRIEVER span and the token usage; then open the trace in the MLflow UI and read the prompt in the `ChatOllama` span.
Did retrieval or the model decide the quality of this answer? (Exercise 2.2 asked the same of printed outputs; here the trace keeps them for you.)

</div>

In [7]:
RIDGE_QUESTION = eval_by_id["q09"]["question"]

print(rag_chain.invoke(RIDGE_QUESTION))

The difference between Ridge and Lasso regularization lies in how they penalize the coefficients of the model.

**Ridge Regression**:
- Uses an L2 penalty, which shrinks coefficients towards zero but does not set them exactly to zero.
- This helps in reducing the complexity of the model and prevents overfitting by penalizing large coefficient values.
- It tends to spread the influence of the data across many features, leading to a more robust model.

**Lasso Regression**:
- Uses an L1 penalty, which can set coefficients exactly to zero.
- This performs feature selection by eliminating features that do not contribute to the model's prediction.
- It can lead to a model that is more interpretable, as only the most important features remain.

**Source**: `lec_11e_polynomial_regression.md`


In [8]:
trace = mlflow.get_trace(mlflow.get_last_active_trace_id(), flush=True)
retriever_span = trace.search_spans(span_type=SpanType.RETRIEVER)[0]
sources = [doc["metadata"]["source"] for doc in retriever_span.outputs]

print(f"sources: {sources}")
print(f"tokens: {trace.info.token_usage}")

sources: ['lecture_11/lec_11e_polynomial_regression.md', 'lecture_11/lec_11e_polynomial_regression.md', 'lecture_13/lec_13a_gridsearchcv_hyperparameter_tuning.md', 'lecture_11/lec_11e_polynomial_regression.md']
tokens: {'input_tokens': 875, 'output_tokens': 166, 'total_tokens': 1041}


<div style="text-align: justify; max-width: 90ch">

**Example answer.** Retrieval decides, and the RETRIEVER span shows how. The expected file of q09 is `lec_11e`, the notebook with the L1/L2 primer and Ridge and Lasso.
If a chunk of it is among the sources, the model can answer from it and name it; if not, a good model says that the notes do not contain the answer, and a fluent answer comes from its training.
Check which case your trace shows, and whether the file the answer names is among the sources. `input_tokens` is the prompt with its four chunks:
how much the model read, not whether it answers.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.3 [G3] A scorer of your own (15 min)

Write a scorer with `@scorer` for a behaviour of your choice, for example "an out-of-scope question is answered with a refusal":
records of questions the notes do not cover carry an expectation that says so, and the scorer abstains (`None`) on all other records.
Test it first on two answers you write yourself, where you know the right verdict, then run it in an evaluation on three out-of-scope questions of your own.

</div>

In [9]:
@scorer
def refuses_out_of_scope(outputs, expectations):
    """An out-of-scope question must get a refusal; abstain on every other record."""
    if not expectations or not expectations.get("out_of_scope"):
        return None  # an in-scope question: another scorer's job

    return is_refusal(outputs)


HAND_WRITTEN = {
    "refusal": "The notes do not contain information about regular expressions.",
    "answer from memory": "A regular expression is a text pattern; Python's re module matches it.",
}

for label, text in HAND_WRITTEN.items():
    verdict = refuses_out_of_scope(outputs=text, expectations={"out_of_scope": True})
    print(f"{label}: {verdict}")

in_scope_verdict = refuses_out_of_scope(
    outputs="...", expectations=records_19[0]["expectations"]
)
print(f"in-scope record: {in_scope_verdict}")

refusal: True
answer from memory: False
in-scope record: None


In [10]:
OUT_OF_SCOPE = [
    "Which lecture covers regular expressions?",
    "Which lecture explains recursion?",
    "How do I scrape a web page with BeautifulSoup?",
]
records_out = [
    {"inputs": {"question": question}, "expectations": {"out_of_scope": True}}
    for question in OUT_OF_SCOPE
]

with mlflow.start_run(run_name="3.3 out of scope"):
    results_out = mlflow.genai.evaluate(
        data=records_out, predict_fn=predict, scorers=[refuses_out_of_scope]
    )

frame = results_out.result_df
for request, verdict, response in zip(
    frame["request"], frame["refuses_out_of_scope/value"], frame["response"]
):
    print(f"{verdict} | {question_of(request)}\n    {response[:150]!r}\n")

2026/10/12 19:09:37 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


True | Which lecture covers regular expressions?
    'The notes do not contain any information about regular expressions. \n\nSource file: lecture_11e_polynomial_regression.md'

True | Which lecture explains recursion?
    'The notes do not contain information about which lecture explains recursion.\n\nSource file: lecture_06/lec_06b_define_functions.md'

True | How do I scrape a web page with BeautifulSoup?
    'The question asks about scraping a web page using BeautifulSoup, but the provided notes do not contain any information about web scraping or Beautiful'



<div style="text-align: justify; max-width: 90ch">

**Example answer.** The two hand-written answers get `True` and `False`, and the in-scope record gets `None`, on every run and without a model.
On the three out-of-scope questions, read each verdict against its answer; in example runs the model refused all three, in phrases of `REFUSAL_PHRASES` (`lec_03a`), which `is_refusal` checks.
Two things to look for:

- **A refusal in other words** ("I'm sorry, the course does not discuss...") would score `False` although the behaviour is right:
  the phrase list is part of the scorer and needs the same review as the evaluation set.
- **A refusal that still names a source file** (look for a source line after the refusal) passes this scorer although no file supports it.
  One scorer checks one behaviour; that one needs a second scorer.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.4 [G4] The judge against the keywords on all 19 (45 min)

Grade the 19 answers of exercise 3.1 with the facts judge of `lec_03b` (`make_facts_judge`) and with `fact_keywords_found`,
without generating new answers: records that carry `outputs` need no `predict_fn`, so both scorers read exactly the same answers.
List the questions where the two disagree (the keywords find every fact but the judge says `no`, or the other way round) and read the judge's rationale for each.
Finally, check with `judge_input_tokens` that every judge prompt fit Ollama's default window (4,096 tokens unless the GPU has 24 GB of VRAM or more,
`lec_03c` section 6).

</div>

In [11]:
states_facts = make_facts_judge(CHAT_MODEL, OLLAMA_HOST)

record_of = {record["inputs"]["question"]: record for record in records_19}
answered_19 = [
    {**record_of[question_of(request)], "outputs": response}
    for request, response in zip(
        results_19.result_df["request"], results_19.result_df["response"]
    )
]

with mlflow.start_run(run_name="3.4 judge vs keywords"):
    results_judge = mlflow.genai.evaluate(
        data=answered_19, scorers=[fact_keywords_found, states_facts]
    )

judged = per_question_table(
    results_judge, eval_set, [fact_keywords_found, states_facts]
)
disagree = judged[
    (judged["fact_keywords_found"] == 1) != (judged["states_facts"] == "yes")
]

print(f"{len(disagree)} of {len(judged)} answers graded differently")
disagree

6 of 19 answers graded differently


,fact_keywords_found,states_facts
id,,
q04,0.500000,yes
q06,0.500000,yes
q10,0.666667,yes
q11,0.500000,yes
q12,0.500000,yes
q13,0.500000,yes


In [12]:
rationale_of = {
    id_of[question_of(request)]: rationale
    for request, rationale in zip(
        results_judge.result_df["request"],
        results_judge.result_df["states_facts/rationale"],
    )
}

for qid in disagree.index:
    print(
        f"{qid}: keywords {disagree.loc[qid, 'fact_keywords_found']:.2f}, judge {disagree.loc[qid, 'states_facts']}"
    )
    print(f"  facts: {eval_by_id[qid]['expectations']['expected_facts']}")
    print(f"  judge: {rationale_of[qid][:300]}\n")

q04: keywords 0.50, judge yes
  facts: ['Call mlflow.log_metric.', 'Inside a run opened with `with mlflow.start_run():`.']
  judge: The answer correctly states that the function to log a metric in an MLflow run is `mlflow.log_metrics()`, which aligns with the expected fact 'Call mlflow.log_metric.' The answer also mentions that the metrics are logged inside a run opened with `with mlflow.start_run():`, which matches the expected

q06: keywords 0.50, judge yes
  facts: ['Run mlflow models serve in a separate terminal.', 'Send POST requests to the /invocations endpoint.']
  judge: The answer correctly addresses the question by providing steps to serve an MLflow model over REST. It mentions using the `mlflow models serve` command, which is the correct approach. The answer also states that POST requests should be sent to the `/invocations` endpoint, which aligns with the expect

q10: keywords 0.67, judge yes
  facts: ['KNN measures nearness with a distance, Euclidean by default.', 'The fea

In [13]:
judge_tokens = judge_input_tokens(results_judge, "states_facts")

print(f"largest judge prompt: {max(judge_tokens)} tokens")

largest judge prompt: 510 tokens


<div style="text-align: justify; max-width: 90ch">

**Example answer.** In example runs the two disagreed on 4 to 6 of 19 answers, all in the same direction: keywords below 1, judge `yes`.
Read each rationale against the facts printed above it. Where the answer states a fact in other words, the judge is right and the keywords miss a paraphrase,
the weakness shown in `lec_03b` section 1. Where the rationale only repeats the expected facts back, it is no evidence: read the answer itself, which may have been cut at 200 tokens before the fact.
A judge that says `yes` to nearly everything may be lenient rather than accurate; how far to trust it is `lec_03c`. The largest judge prompt was a few hundred tokens in example runs, far below the default window:
no verdict was given on a cut prompt.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.5 [G5] A variant of your own, quality against cost (45 min)

Pick one change to the system: `num_predict` 200 against 400, a change to the prompt (`make_rag_chain(retriever, llm, prompt_template=...)`),
or the rewrite step of `lec_02d` (`make_rag_chain(retriever, llm, rewriter=make_query_rewriter(llm))`). Evaluate both variants on all 19 questions with the same suite,
the deterministic scorers (`answer_tokens` among them), `prompt_tokens` and the facts judge, one named run each, and put them side by side with `compare_variants`.
Which rows of the table measure quality, and which cost? Which variant would you ship?

</div>

In [14]:
llm_400 = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=400,
)
rag_chain_400 = make_rag_chain(retriever, llm_400)


def predict_400(question):
    """Variant num_predict=400: the same chain with room for twice the answer."""
    return rag_chain_400.invoke(question)


SUITE = [*DETERMINISTIC_SCORERS, prompt_tokens, states_facts]
results_by_variant = {}

for name, predict_fn in [
    ("num_predict=200", predict),
    ("num_predict=400", predict_400),
]:
    with mlflow.start_run(run_name=f"3.5 {name}"):
        results_by_variant[name] = mlflow.genai.evaluate(
            data=records_19, predict_fn=predict_fn, scorers=SUITE
        )

2026/10/12 19:09:58 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


2026/10/12 19:10:46 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


In [15]:
compare_variants(results_by_variant).round(2)

,num_predict=200,num_predict=400
answer_tokens/max,200.00,400.00
answer_tokens/mean,161.95,208.42
cites_expected_source/mean,0.58,0.79
fact_keywords_found/mean,0.83,0.83
prompt_tokens/mean,757.16,757.16
refused/mean,0.00,0.00
retrieval_hit/mean,1.00,1.00
seconds_per_answer/mean,15.94,18.00
states_facts/mean,1.00,1.00


In [16]:
tables = {
    name: per_question_table(results, eval_set, SUITE)
    for name, results in results_by_variant.items()
}

pd.DataFrame(
    {
        "tokens_200": tables["num_predict=200"]["answer_tokens"],
        "cites_200": tables["num_predict=200"]["cites_expected_source"],
        "tokens_400": tables["num_predict=400"]["answer_tokens"],
        "cites_400": tables["num_predict=400"]["cites_expected_source"],
    }
)

,tokens_200,cites_200,tokens_400,cites_400
id,,,,
q01,106.0,True,106.0,True
q02,152.0,True,152.0,True
q03,119.0,True,119.0,True
q04,200.0,True,341.0,True
q05,200.0,True,253.0,True
q06,200.0,False,329.0,True
q07,188.0,True,188.0,True
q08,173.0,True,173.0,True
q09,166.0,True,166.0,True


<div style="text-align: justify; max-width: 90ch">

**Example answer.** Quality rows: `cites_expected_source`, `fact_keywords_found`, `states_facts`. Cost rows: `prompt_tokens`, `answer_tokens`, `seconds_per_answer`.
`prompt_tokens` and `retrieval_hit` are the same in both columns on every run: `num_predict` acts after the prompt is built, so the same chunks reach the same prompt.
The cost moves to the output side, `answer_tokens` and seconds. In example runs `cites_expected_source/mean` rose from about 0.5 to about 0.75 or more,
while the judge said `yes` to every answer of both variants and could not tell them apart. The per-question table shows where:
answers cut at 200 tokens before their source line get room for it at 400; answers that still use all 400 tokens are long ones that a larger limit only makes longer.
I would ship 400, then test a prompt that asks for the source line first as the next variant. Nineteen questions are still a small set.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.6 [G6] The judge on answers `lec_03c` did not use (45 min)

`corpus/eval/judge_labels.jsonl` holds 26 labelled answers; `lec_03c` graded eleven of them. Take the other fifteen:

- First label them yourself by the rubric of `lec_03b` (`yes` only if the answer states every expected fact and contradicts none),
  with the `label` column hidden. Then compare with the file's labels: where you differ, is the rubric unclear, or did one of you misread?
- Run the judge (`make_facts_judge`) on the fifteen with `run_judge_on` and measure it against the labels with `measure_agreement_between` (`lec_03c`).
  Is its false-yes rate like the one in `lec_03c`, and which `kind` of answer does it pass wrongly?

</div>

In [17]:
labelled = pd.read_json(REPO / "corpus" / "eval" / "judge_labels.jsonl", lines=True)
USED_IN_LEC_03C = [
    "j02",
    "j04",
    "j06",
    "j13",
    "j14",
    "j15",
    "j16",
    "j18",
    "j21",
    "j22",
    "j26",
]
others = labelled[~labelled["id"].isin(USED_IN_LEC_03C)].reset_index(drop=True)

records_by_id = {
    row["id"]: record
    for row, record in zip(eval_set, build_eval_records_from(eval_set))
}
judge = make_facts_judge(CHAT_MODEL, OLLAMA_HOST)
others = others.assign(verdict=run_judge_on(judge, others, records_by_id))

print(measure_agreement_between(others["label"], others["verdict"]).round(2))

agreement         0.73
false_yes_rate    0.57
false_no_rate     0.00
dtype: float64


In [18]:
others[["id", "kind", "label", "verdict"]]

,id,kind,label,verdict
0,j01,right,yes,yes
1,j03,reversed,no,no
2,j05,right,yes,yes
3,j07,partial,no,no
4,j08,paraphrase,yes,yes
5,j09,reversed,no,yes
6,j10,wrong detail,no,yes
7,j11,paraphrase,yes,yes
8,j12,reversed,no,yes
9,j17,right,yes,yes


<div style="text-align: justify; max-width: 90ch">

**Example answer.** Your own labels first: the file's labels are a draft (`lec_03c`, section 2). Where you and the file differ, decide by the rubric and write down why;
a disagreement between two careful readers marks an unclear expected fact, and no judge can do better than the rubric.

The judge against the file's labels: in one example run it agreed on 11 of the 15 (0.73) with no false `no`, and passed 4 of the 7 answers labelled `no` (a false-yes rate of 0.57),
about as lenient as in `lec_03c`. The rows show which: in that run it passed two reversed answers and two with a wrong detail,
and caught a reversed, a partial and a real answer. Fifteen answers are few: one verdict moves the false-yes rate by about 0.14.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.7 [G7] A tool of your own (45 min)

Write a read-only tool with `@tool` that the three tools of `lec_03d` lack, for example the number of words of one notebook,
given its file name. It must check its argument and **return** an error the model can read, not raise. `run_tool_calls(reply, tools)` (`course_helpers.agents`) runs every tool call of one model reply and returns one `ToolMessage` per call,
an error message for a bad call. `message_table(messages)` gives one row per message: its role, its tool calls and its text.
Then:

- run the loop of `lec_03d` section 3 by hand for a question your tool answers, and print the conversation with `message_table`;
- write a bad model reply yourself, an `AIMessage` whose `tool_calls` hold a wrong value, a wrong type and an unknown tool name,
  and run it with `run_tool_calls`. What does each error tell the model?

</div>

In [19]:
file_of = {Path(d.metadata["source"]).name: d for d in documents}


@tool
def count_words(file_name: str) -> dict:
    """Count the words of one file of the course notes.

    Give the file name with its extension, such as "lec_04b_index_slice_list_tuple_set.md".
    """
    if file_name not in file_of:
        # Suggest the files of the same notebook prefix, e.g. every "lec_04b..." file
        similar = sorted(name for name in file_of if name[:7] == file_name[:7])
        return {
            "error": f"There is no file {file_name!r}; files starting the same: {similar}."
        }
    return {
        "file_name": file_name,
        "words": len(file_of[file_name].page_content.split()),
    }


print(count_words.invoke({"file_name": "lec_04b_index_slice_list_tuple_set.md"}))
print(count_words.invoke({"file_name": "lec_04b_tuples.md"}))

{'file_name': 'lec_04b_index_slice_list_tuple_set.md', 'words': 1516}
{'error': "There is no file 'lec_04b_tuples.md'; files starting the same: ['lec_04b_index_slice_list_tuple_set.md']."}


In [20]:
WORDS_QUESTION = (
    "How many words does the notebook lec_04b_index_slice_list_tuple_set.md have?"
)

llm_with_count = llm.bind_tools([count_words])
messages = [HumanMessage(WORDS_QUESTION)]

for step in range(3):  # a hard limit, as in every agent
    reply = llm_with_count.invoke(messages)
    messages.append(reply)
    if not reply.tool_calls:
        break
    messages.extend(run_tool_calls(reply, [count_words]))

# Every row in full: a table would cut the file name and the count short
for row in message_table(messages, width=300):
    print(f"{row['role']:>5} | {row['tool calls']} | {row['content']}")

human |  | How many words does the notebook lec_04b_index_slice_list_tuple_set.md have?
   ai | count_words({'file_name': 'lec_04b_index_slice_list_tuple_set.md'}) | 
 tool |  | {"file_name": "lec_04b_index_slice_list_tuple_set.md", "words": 1516}
   ai |  | The notebook "lec_04b_index_slice_list_tuple_set.md" contains 1516 words.


In [21]:
# A reply written by hand: three calls a model could have written
bad_reply = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "count_words",
            "args": {"file_name": "lec_04b_tuples.md"},
            "id": "wrong-value",
        },
        {"name": "count_words", "args": {"file_name": 42}, "id": "wrong-type"},
        {
            "name": "count_lines",
            "args": {"file_name": "lec_04b.md"},
            "id": "unknown-tool",
        },
    ],
)

for message in run_tool_calls(bad_reply, [count_words]):
    print(f"{message.tool_call_id}: {message.content}\n")

wrong-value: {"error": "There is no file 'lec_04b_tuples.md'; files starting the same: ['lec_04b_index_slice_list_tuple_set.md']."}

wrong-type: Error: ValidationError: 1 validation error for count_words file_name Input should be a valid string [type=string_type, input_value=42, input_type=int]

unknown-tool: Error: there is no tool 'count_lines'; the tools are ['count_words'].



<div style="text-align: justify; max-width: 90ch">

**Example answer.** The two direct calls print a count and an error with the files that start like the wrong name, on every run.
In the loop: did the model pass the full file name, and does its last message give the count of the direct call? The schema is the tool's prompt, names included:
an argument name and a docstring that disagree (an argument `notebook` described as a "file name") make the model write the wrong key,
and the call fails validation before the tool runs. The three hand-written calls:

- **Wrong value**: your own check returns an error with names to try.
- **Wrong type**: pydantic rejects `42` for a `str` before the function runs.
- **Unknown tool**: `run_tool_calls` lists the tools that exist.

Each comes back as a `tool` message: the model can correct its call, and the program does not stop.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.8 [G8] Thinking on and off for a tool choice (15 min)

Bind the course's tools (`make_course_tools(documents, retriever)`) to the chat model and ask the question of `lec_03d` section 6,
"Do lectures 9 and 13 have the same number of lecture notebooks?", in single calls: five with thinking off and five with thinking on, at the same temperature.
For each call record whether it asks `list_notebooks` for both lectures, its output tokens and its seconds. Does thinking help this decision, and at what price?

</div>

In [22]:
COMPARE_QUESTION = "Do lectures 9 and 13 have the same number of lecture notebooks?"
course_tools = make_course_tools(documents, retriever)
rows = []

for thinking in [False, True]:
    # Same temperature for both (the Qwen3 card's 0.6 for thinking): only thinking differs
    model = ChatOllama(
        model=CHAT_MODEL,
        base_url=OLLAMA_HOST,
        reasoning=thinking,
        temperature=0.6,
        num_predict=2000,  # thinking tokens count against it
    ).bind_tools(course_tools)

    for run in range(1, 6):
        started = time.time()
        reply = model.invoke(COMPARE_QUESTION)
        lectures = {
            call["args"].get("lecture")
            for call in reply.tool_calls
            if call["name"] == "list_notebooks"
        }
        rows.append(
            {
                "thinking": thinking,
                "run": run,
                "asks for 9 and 13": lectures == {9, 13},
                "tool calls": len(reply.tool_calls),
                "output tokens": reply.usage_metadata["output_tokens"],
                "seconds": round(time.time() - started, 1),
            }
        )

runs_36 = pd.DataFrame(rows)

runs_36

,thinking,run,asks for 9 and 13,tool calls,output tokens,seconds
0,False,1,True,2,43,0.4
1,False,2,False,1,70,0.6
2,False,3,False,0,231,2.1
3,False,4,False,0,155,1.4
4,False,5,False,0,213,1.9
5,True,1,True,2,354,6.7
6,True,2,True,2,282,2.6
7,True,3,True,2,199,1.8
8,True,4,True,2,212,1.9
9,True,5,True,2,279,2.5


In [23]:
runs_36.groupby("thinking")[
    ["asks for 9 and 13", "output tokens", "seconds"]
].mean().round(2)

,asks for 9 and 13,output tokens,seconds
thinking,,,
False,0.2,142.4,1.28
True,1.0,265.2,3.10


<div style="text-align: justify; max-width: 90ch">

**Example answer.** Read the `asks for 9 and 13` column. In example runs thinking asked for both lectures in 5 of 5 calls and no thinking in 0 to 3 of 5 (the others asked for one lecture or none),
in line with the example runs of `lec_03d`. In example runs thinking cost about twice the output tokens and seconds per call, or more.
Five calls per setting can show a large difference, not measure a rate: a decision that fails one time in five may well not fail in five calls.
Thinking pays where the model makes a decision and a wrong one is expensive (a missing call means an answer from memory);
for a step that must always happen, code is cheaper and never skips it.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.9 [O1] The chain against the graph on all 19 and your own wording (90 min)

Write three questions of your own that reword questions of the evaluation set the way a student might, without the notes' terms (as `lec_03e` section 1 did for q19,
and `lec_02d` section 7 for six questions), and give each the expectations of its original. Evaluate the plain chain and the graph of `lec_03e` (`make_agentic_rag`) on the 19 questions plus yours,
with the deterministic scorers, `searches`, `total_tokens` and the facts judge of `lec_03b`, both with `num_predict=300` as in `lec_03e`.
Compare the aggregates, then the rows where the two systems differ. End with a decision record in the format of `lec_03e` section 8:
is the loop worth its price for this course?

</div>

In [24]:
REWORDED = {
    "q19": "Why can't I change an item of something written in round brackets?",
    "q01": "My test score is suspiciously good after I scaled all the data first, what went wrong?",
    "q17": "What is the strange line with double underscores at the bottom of scripts for?",
}
questions_37 = eval_set + [
    {**eval_by_id[qid], "id": qid + "r", "question": text}
    for qid, text in REWORDED.items()
]
records_37 = build_eval_records_from(questions_37)

llm_300 = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=300,
)
rag_chain_300 = make_rag_chain(retriever, llm_300)
graph = make_agentic_rag(retriever, llm_300, max_searches=2)

SYSTEMS = {
    "plain chain": lambda question: rag_chain_300.invoke(question),
    "agentic graph": lambda question: graph.invoke({"question": question})["answer"],
}
SUITE_37 = [*DETERMINISTIC_SCORERS, searches, total_tokens, states_facts]
results_by_system = {}

for name, predict_fn in SYSTEMS.items():
    with mlflow.start_run(run_name=f"3.9 {name}"):
        results_by_system[name] = mlflow.genai.evaluate(
            data=records_37, predict_fn=predict_fn, scorers=SUITE_37
        )

2026/10/12 19:12:17 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


2026/10/12 19:13:22 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


In [25]:
compare_variants(results_by_system).round(2)

,plain chain,agentic graph
answer_tokens/max,300.00,327.00
answer_tokens/mean,202.14,232.00
cites_expected_source/mean,0.73,0.82
fact_keywords_found/mean,0.77,0.80
refused/mean,0.00,0.00
retrieval_hit/mean,0.91,0.91
searches/mean,1.00,1.00
seconds_per_answer/mean,24.81,34.48
states_facts/mean,0.95,0.95
total_tokens/mean,961.86,1802.14


In [26]:
COLUMNS = [
    "retrieval_hit",
    "cites_expected_source",
    "states_facts",
    "searches",
    "total_tokens",
]
both = (
    pd.concat(
        {
            name: per_question_table(results, questions_37, SUITE_37)[COLUMNS]
            for name, results in results_by_system.items()
        },
        names=["system", "id"],
    )
    .swaplevel()
    .sort_index(level="id", sort_remaining=False)
)

# Keep the questions where the systems differ on the right file or the judge, and the reworded ones
quality = both[["retrieval_hit", "states_facts"]].astype(str)
differ = quality.groupby(level="id").nunique().gt(1).any(axis=1)
shown = [qid for qid in differ.index if differ[qid] or qid.endswith("r")]

both.loc[shown]

retrieval_hit  cites_expected_source states_facts  \
id   system                                                             
q01r plain chain            False                  False          yes   
     agentic graph          False                  False          yes   
q17r plain chain            False                  False           no   
     agentic graph          False                  False           no   
q19r plain chain             True                   True          yes   
     agentic graph           True                   True          yes   

                    searches  total_tokens  
id   system                                 
q01r plain chain         1.0         945.0  
     agentic graph       1.0        1791.0  
q17r plain chain         1.0         972.0  
     agentic graph       1.0        1914.0  
q19r plain chain         1.0        1110.0  
     agentic graph       1.0        1825.0

<div style="text-align: justify; max-width: 90ch">

**Example answer.** On the 19 original questions both systems found the expected file equally often in example runs: the questions' own words find it.
In example runs the graph nearly doubled `total_tokens/mean`; if `searches/mean` is 1.0 in your run, the retry never ran.
On the reworded rows, whether the chain finds the file depends on the index and the embedding model: in earlier example runs it missed all three, in a later one it found q19r's `lec_04b`.
Where the chain misses, the graph hits only if the model's query supplies the notes' term; in example runs of this notebook the model often copied the student's words into the query,
although in `lec_03e`'s example runs it wrote *tuple* every time. Look for a judge `yes` without the file: an answer from memory.

</div>

<div style="text-align: justify; max-width: 90ch">

**Example decision record.**

- **Decision:** if the graph's `retrieval_hit` on the reworded rows is no better than the chain's, the plain chain; if it hits where the chain misses, the rewrite step, without the retry.
- **Requirements:** the right notes in the prompt; the file named; tokens and seconds per answer.
- **Evidence:** this evaluation, 22 questions, one run per system.
- **Worth it when:** students' words differ from the notes' and the rewrite fixes it reliably.
- **Not worth it when:** the questions already find their files; cost or latency matter most.
- **Revisit when:** the questions, the notes or the model change.

</div>

<div style="text-align: justify; max-width: 90ch">

### Exercise 3.10 [O1] A strict grade against the topic grade (45 min)

`lec_03e` grades the chunks with a question about the **topic** (`GRADE_PROMPT` in `course_helpers.agents`). Write a strict grade prompt that asks whether the notes contain the **answer**,
with the same `GRADE_SCHEMA`, and grade the same searches with both prompts:

- the 19 questions in their own words, where the expected file is found;
- your reworded questions of exercise 3.9, which may miss it;
- the out-of-scope questions of exercise 3.3.

In the graph a `no` costs one more search. Count where each prompt says `no` to chunks that hold the expected file, and `yes` to an out-of-scope question.
Which grade would you put in the graph?

</div>

In [27]:
STRICT_GRADE_PROMPT = (
    "Notes:\n{context}\n\nQuestion: {question}\n\n"
    "Reply yes only if the notes above state the answer to this exact question; "
    "otherwise reply no."
)
grade_chains = {
    name: ChatPromptTemplate.from_template(prompt)
    | llm.with_structured_output(GRADE_SCHEMA, method="json_schema")
    for name, prompt in [("topic", GRADE_PROMPT), ("strict", STRICT_GRADE_PROMPT)]
}

cases = [
    (
        q["id"],
        "reworded" if q["id"].endswith("r") else "own words",
        q["question"],
        q["expected_source"],
    )
    for q in questions_37
]
cases += [
    (f"out{i}", "out of scope", text, None) for i, text in enumerate(OUT_OF_SCOPE, 1)
]
rows = []

for case_id, kind, question, expected in cases:
    found = retriever.invoke(question)
    row = {
        "id": case_id,
        "kind": kind,
        "file found": None
        if expected is None
        else expected in [d.metadata["source"] for d in found],
    }
    context = format_context_from(found)
    for name, chain in grade_chains.items():
        row[name] = chain.invoke({"context": context, "question": question})[
            "topic_explained"
        ]
    rows.append(row)

grades = pd.DataFrame(rows).set_index("id")

grades[grades["topic"] != grades["strict"]]

,kind,file found,topic,strict
id,,,,
out3,out of scope,None,no,yes


In [28]:
summary = {}

for name in grade_chains:
    summary[name] = {
        "no, file found (a wasted search)": (
            grades["file found"].eq(True) & grades[name].eq("no")
        ).sum(),
        "no, file missed (a useful retry)": (
            grades["file found"].eq(False) & grades[name].eq("no")
        ).sum(),
        "yes, out of scope (answers at once)": (
            grades["kind"].eq("out of scope") & grades[name].eq("yes")
        ).sum(),
    }

pd.DataFrame(summary)

,topic,strict
"no, file found (a wasted search)",0,0
"no, file missed (a useful retry)",0,0
"yes, out of scope (answers at once)",0,1


<div style="text-align: justify; max-width: 90ch">

**Example answer.** The first table lists only the searches where the two prompts disagree; in one example run that was a single row,
the round-brackets question, where the strict grade said `no` to chunks without `lec_04b` and the topic grade said `yes`.
The summary prices each verdict in the graph:

- **`no` on chunks with the expected file**: a wasted search.
- **`no` on a near miss**: a useful retry, if the second query finds the file.
- **`yes` out of scope**: the graph answers at once, and the answer prompt must refuse.

In that run the strict grade made the one useful retry and wasted no search, unlike the stricter wording of `lec_03e`'s example runs;
in another, the single row was an out-of-scope question, where the strict grade said `yes` and the topic grade `no`, so the strict grade let the graph answer at once.
A 1.7b grader can err either way from run to run.

</div>

<div style="text-align: justify; max-width: 90ch">

Before switching, I would build the graph of `lec_03e` with each grade and run both on the reworded questions of 3.9: these counts rest on 25 searches with one verdict each.

</div>